# Chest X-Ray Classification - Training Pipeline

**Project:** Pneumonia Detection using Deep Learning  
**Model:** MobileNetV2 Transfer Learning  
**Objective:** Train a binary classifier to distinguish NORMAL from PNEUMONIA chest X-rays using preprocessed float32 .npy files.

## Table of Contents
1. **Environment & GPU Setup** - Verify hardware and configure TensorFlow
2. **Configuration** - All hyperparameters in one place
3. **Data Loading** - Load float32 .npy files and verify z-score preservation
4. **Class Weights** - Handle class imbalance without augmentation
5. **Model Architecture** - MobileNetV2 transfer learning
6. **Phase 1 Training** - Train head only (base frozen)
7. **Phase 2 Fine-Tuning** - Unfreeze last 14 layers
8. **Training Curves** - Visualise accuracy, loss, AUC
9. **Evaluation** - Test set metrics, threshold sweep, confusion matrix
10. **Sample Predictions** - Visual check on test images

## 1. Environment & GPU Setup

Here we load all required libraries and configure TensorFlow to use GPU memory growth. Without `set_memory_growth`, TensorFlow tries to allocate all available VRAM at startup, which can crash other applications sharing the GPU.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

# GPU SETUP
# enable memory growth so TensorFlow doesn't grab all VRAM at once
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU found: {gpus}")
else:
    print("No GPU — running on CPU")

## 2. Configuration

All hyperparameters are defined here. Change any value in this cell only — the rest of the notebook reads from these variables.

In [ ]:
SCRIPT_DIR = Path(".")

# paths
PROCESSED_DIR   = str((SCRIPT_DIR / '../../chest_xray/train_processed/').resolve())
TEST_DIR        = str((SCRIPT_DIR / '../../chest_xray/test_processed/').resolve())
MODEL_SAVE_PATH = str(SCRIPT_DIR / 'xray_mobilenet_model.keras')
LOG_DIR         = str(SCRIPT_DIR / 'training_logs/')

# settings
IMG_SIZE     = (128, 128)
IMG_CHANNELS = 1         

# training - phase 1 (frozen base, train head only)
BATCH_SIZE    = 32
EPOCHS        = 20        
LEARNING_RATE = 1e-4       # standard Adam LR for new head

# fine-tuning — phase 2
FINE_TUNE_EPOCHS   = 10    # fewer epochs to avoid overfitting
FINE_TUNE_LR       = 5e-6  # very low - avoids destroying pretrained weights
FINE_TUNE_AT_LAYER = 140   # unfreeze last 14 of 154 MobileNetV2 layers
                            
VAL_SPLIT = 0.2

# regularization
L2_LAMBDA     = 1e-3       # 10x stronger than initial 1e-4
DROPOUT_DENSE = 0.60
DENSE_UNITS   = 64         # small head — fewer parameters to memorise

# callbacks
EARLY_STOP_PATIENCE = 5
REDUCE_LR_PATIENCE  = 3
REDUCE_LR_FACTOR    = 0.5
REDUCE_LR_MIN       = 1e-7

# decision threshold — 0.70 balances NORMAL recall 0.72 and PNEUMONIA recall 0.93
DECISION_THRESHOLD = 0.70

os.makedirs(LOG_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  Processed dir : {PROCESSED_DIR}")
print(f"  Test dir      : {TEST_DIR}")
print(f"  Model save    : {MODEL_SAVE_PATH}")

## 3. Data Loading

We load `.npy` float32 files directly instead of using `ImageDataGenerator`.

**Why .npy and not PNG?**

Previous iterations saved preprocessed images as uint8 PNG. PNG save requires rescaling float32 back to 0–255, which silently undoes z-score normalisation and restores absolute brightness. This caused a **domain shift of 26.7 intensity units** between train NORMAL (mean 124.9) and test NORMAL (mean 151.6). The model learned brightness as a class proxy and consistently misclassified bright test NORMAL images as PNEUMONIA.

Loading float32 `.npy` preserves exact values. Domain shift drops to < 0.1 units.

In [ ]:
def load_npy_dataset(base_dir, val_split=0.0, subset='all', seed=42):
    """
    Load all .npy files from base_dir/NORMAL and base_dir/PNEUMONIA.
    Returns numpy arrays (X, y) — float32 images and binary labels.
    subset: 'all', 'training', 'validation'
    """
    X, y = [], []

    for label, class_name in enumerate(['NORMAL', 'PNEUMONIA']):
        folder = os.path.join(base_dir, class_name)
        if not os.path.exists(folder):
            print(f"  WARNING: {folder} not found")
            continue
        files = sorted([f for f in os.listdir(folder) if f.endswith('.npy')])
        print(f"  {class_name}: {len(files)} files")
        for fname in files:
            arr = np.load(os.path.join(folder, fname))
            X.append(arr)
            y.append(label)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    # added channel dimension: (N, 128, 128) -> (N, 128, 128, 1)
    X = X[..., np.newaxis]

    rng  = np.random.default_rng(seed)
    idx  = rng.permutation(len(X))
    X, y = X[idx], y[idx]

    if val_split > 0 and subset in ('training', 'validation'):
        split_at = int(len(X) * (1 - val_split))
        if subset == 'training':
            return X[:split_at], y[:split_at]
        else:
            return X[split_at:], y[split_at:]

    return X, y

In [ ]:
print("Loading TRAIN data...")
X_train, y_train = load_npy_dataset(PROCESSED_DIR, val_split=VAL_SPLIT, subset='training')
X_val,   y_val   = load_npy_dataset(PROCESSED_DIR, val_split=VAL_SPLIT, subset='validation')

print("\nLoading TEST data...")
X_test, y_test = load_npy_dataset(TEST_DIR)

print(f"\nShapes:")
print(f"  Training   : {X_train.shape}")
print(f"  Validation : {X_val.shape}")
print(f"  Test       : {X_test.shape}")

In [ ]:
# Verify z-score preserved — mean ~0, std ~1
print("Z-score verification:")
print(f"  Train  mean: {X_train.mean():.3f}  std: {X_train.std():.3f}")
print(f"  Val    mean: {X_val.mean():.3f}  std: {X_val.std():.3f}")
print(f"  Test   mean: {X_test.mean():.3f}  std: {X_test.std():.3f}")

# Domain shift check
train_normal_mean = X_train[y_train == 0].mean()
test_normal_mean  = X_test[y_test   == 0].mean()
diff = abs(train_normal_mean - test_normal_mean)

print(f"\nDomain shift check:")
print(f"  Train NORMAL mean : {train_normal_mean:.3f}")
print(f"  Test  NORMAL mean : {test_normal_mean:.3f}")
print(f"  Difference        : {diff:.3f}  ({'OK' if diff < 0.1 else 'WARNING'})")

## 4. Class Weights

Training set: 1,341 NORMAL vs 3,875 PNEUMONIA (ratio 0.35).

**Why not augmentation?**

Augmentation was tested first. 2,534 synthetic NORMAL images were generated. NORMAL recall dropped to 0.39 because 65% of NORMAL training images became artificial — the model learned augmentation artefacts not real lung anatomy.

`compute_class_weight('balanced')` gives NORMAL ~1.95 and PNEUMONIA ~0.67. Each NORMAL misclassification counts ~2.9x more during gradient updates - no fake data required.

In [ ]:
class_weights_array = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.array([0, 1]),
    y            = y_train
)
class_weight_dict = {
    0: class_weights_array[0],   # NORMAL — higher weight
    1: class_weights_array[1]    # PNEUMONIA — lower weight
}

print("Class weights:")
print(f"  NORMAL    (0) : {class_weight_dict[0]:.4f}")
print(f"  PNEUMONIA (1) : {class_weight_dict[1]:.4f}")

## 5. Model Architecture — MobileNetV2 Transfer Learning

**Why MobileNetV2 instead of a custom CNN?**

A custom CNN trained from scratch achieved a maximum NORMAL recall of 0.52. MobileNetV2 was pretrained on 1.4 million ImageNet images and already understands edges, textures, and structural patterns. Only the final NORMAL vs PNEUMONIA distinction needs to be learned.

```
Input (128x128x1 grayscale float32)
    |
[1x1 Conv2D]  -- grayscale (1ch) to RGB (3ch)
    |
[MobileNetV2 base]  -- 154 pretrained layers, frozen in Phase 1
    |
[GlobalAveragePooling2D]  -- 4x4x1280 -> 1280 values
    |
[Dense 64, ReLU, L2=1e-3]
    |
[Dropout 0.60]
    |
[Dense 1, Sigmoid]  -- P(PNEUMONIA)
```

In [ ]:
def build_mobilenet(input_shape=(128, 128, 1)):

    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(3, (1, 1), padding='same', name='grayscale_to_rgb')(inputs)

    # MobileNetV2 pretrained on ImageNet
    # include_top=False removes the original 1000-class head
    base_model = keras.applications.MobileNetV2(
        input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3),
        include_top = False,
        weights     = 'imagenet'
    )
    base_model.trainable = False   # phase 1: all frozen

    # training=False keeps BatchNorm in inference mode when base is frozen
    x = base_model(x, training=False)

    # global average pooling: 4x4x1280 -> 1280 values
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(DENSE_UNITS, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(L2_LAMBDA))(x)
    x = layers.Dropout(DROPOUT_DENSE)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model, base_model


model, base_model = build_mobilenet(input_shape=(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS))
model.summary()

print(f"\nMobileNetV2 total layers  : {len(base_model.layers)}")
print(f"Frozen in Phase 1         : all ({len(base_model.layers)})")
print(f"Unfrozen in Phase 2       : {len(base_model.layers) - FINE_TUNE_AT_LAYER}")

## 6. Phase 1 - Train Head Only (Base Frozen)

Only Dense(64) + Dropout + Sigmoid train here. MobileNetV2 base is fully frozen. This phase learns basic NORMAL vs PNEUMONIA separation without touching pretrained weights.

In [ ]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

def get_callbacks(suffix=''):
    return [
        callbacks.ModelCheckpoint(
            filepath             = MODEL_SAVE_PATH,
            monitor              = 'val_auc',
            save_best_only       = True,
            mode                 = 'max',
            verbose              = 1
        ),
        callbacks.EarlyStopping(
            monitor              = 'val_auc',
            patience             = EARLY_STOP_PATIENCE,
            restore_best_weights = True,
            mode                 = 'max',
            verbose              = 1
        ),
        callbacks.ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = REDUCE_LR_FACTOR,
            patience = REDUCE_LR_PATIENCE,
            min_lr   = REDUCE_LR_MIN,
            verbose  = 1
        ),
        callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=0),
        callbacks.CSVLogger(
            os.path.join(LOG_DIR, f'training_history{suffix}.csv'),
            append=False
        )
    ]

In [ ]:
print("-" * 55)
print("PHASE 1 — TRAINING HEAD (base frozen)")
print("-" * 55)

history_phase1 = model.fit(
    X_train, y_train,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_data = (X_val, y_val),
    callbacks       = get_callbacks(suffix='_phase1'),
    class_weight    = class_weight_dict,
    verbose         = 1
)

print(f"\nPhase 1 complete. Best model: {MODEL_SAVE_PATH}")

## 7. Phase 2 - Fine-Tuning (Last 14 Layers)

Layers 140–154 of MobileNetV2 are unfrozen. These learn chest X-ray specific features.

**Why only 14 layers?** A previous attempt with 54 unfrozen layers and LR=1e-5 caused severe overfitting: train loss 0.023 vs test loss 2.8. Limiting to 14 layers and LR=5e-6 resolved this.

In [ ]:
print("-" * 55)
print(f"PHASE 2 — FINE-TUNING (last {len(base_model.layers) - FINE_TUNE_AT_LAYER} layers)")
print("-" * 55)

base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT_LAYER]:
    layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"MobileNetV2 trainable layers: {trainable} / {len(base_model.layers)}")

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

history_phase2 = model.fit(
    X_train, y_train,
    epochs          = FINE_TUNE_EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_data = (X_val, y_val),
    callbacks       = get_callbacks(suffix='_phase2'),
    class_weight    = class_weight_dict,
    verbose         = 1
)

print(f"\nPhase 2 complete. Best model: {MODEL_SAVE_PATH}")

## 8. Training Curves

Phase 1 and Phase 2 histories concatenated into one continuous plot. **Green dashed line** marks the start of fine-tuning.

Healthy training shows train and val curves tracking closely with no large gap, and loss decreasing steadily in both splits.

In [ ]:
def plot_training_history(h1, h2, save_dir=LOG_DIR):
    combined   = {k: h1.history[k] + h2.history[k] for k in h1.history}
    phase1_end = len(h1.history['loss'])

    metrics = [
        ('accuracy',  'val_accuracy',  'Accuracy'),
        ('loss',      'val_loss',      'Loss'),
        ('auc',       'val_auc',       'AUC'),
        ('precision', 'val_precision', 'Precision'),
        ('recall',    'val_recall',    'Recall'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training History — Phase 1 + Phase 2', fontsize=15, fontweight='bold')
    axes = axes.flatten()

    for idx, (tk, vk, title) in enumerate(metrics):
        if tk in combined:
            axes[idx].plot(combined[tk], label='Train', color='steelblue')
            axes[idx].plot(combined[vk], label='Val',   color='tomato')
            axes[idx].axvline(x=phase1_end, color='green',
                              linestyle='--', linewidth=1.5, label='Fine-tune start')
            axes[idx].set_title(title)
            axes[idx].set_xlabel('Epoch')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)

    axes[-1].axis('off')
    plt.tight_layout()
    path = os.path.join(save_dir, 'training_curves.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f"Saved to: {path}")
    plt.show()

plot_training_history(history_phase1, history_phase2)

## 9. Evaluation on Test Set

Best saved model loaded (highest val_auc checkpoint, not last epoch).

**Targets:** PNEUMONIA Recall > 0.95 | NORMAL Recall > 0.70 | AUC > 0.95

In [ ]:
print("-" * 55)
print("EVALUATING ON TEST SET")
print("-" * 55)

best_model   = keras.models.load_model(MODEL_SAVE_PATH)
test_results = best_model.evaluate(X_test, y_test, verbose=1)

print("\nTest Results:")
for name, value in zip(best_model.metrics_names, test_results):
    print(f"  {name:<20} : {value:.4f}")

### Threshold Sweep

Rows marked `<--` satisfy both NORMAL recall ≥ 0.70 and PNEUMONIA recall ≥ 0.90.

In [ ]:
y_pred_prob = best_model.predict(X_test, verbose=1).flatten()
class_names = ['NORMAL', 'PNEUMONIA']

print("\nProbability Distribution:")
print(f"  NORMAL    -- mean: {y_pred_prob[y_test==0].mean():.3f}")
print(f"  PNEUMONIA -- mean: {y_pred_prob[y_test==1].mean():.3f}")

print("\nThreshold | NORMAL recall | PNEUMONIA recall | Accuracy")
print("-" * 58)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (y_pred_prob > t).astype(int)
    nr   = (pred[y_test == 0] == 0).mean()
    pr   = (pred[y_test == 1] == 1).mean()
    acc  = (pred == y_test).mean()
    flag = " <--" if nr >= 0.70 and pr >= 0.90 else ""
    print(f"  {t}     |     {nr:.2f}      |       {pr:.2f}       |  {acc:.2f}{flag}")

### Classification Report

In [ ]:
print("-" * 55)
print(f"THRESHOLD COMPARISON: 0.50 vs {DECISION_THRESHOLD}")
print("-" * 55)

for thresh in [0.50, DECISION_THRESHOLD]:
    y_pred = (y_pred_prob > thresh).astype(int)
    print(f"\nThreshold = {thresh}")
    print(classification_report(y_test, y_pred, target_names=class_names))

### Confusion Matrix

Two matrices side by side: default (0.50) vs selected (0.70) threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrix Comparison', fontsize=13, fontweight='bold')

for ax, thresh, title in zip(
    axes,
    [0.50, DECISION_THRESHOLD],
    ['Default Threshold (0.50)', f'Selected Threshold ({DECISION_THRESHOLD})']
):
    y_p  = (y_pred_prob > thresh).astype(int)
    cm_t = confusion_matrix(y_test, y_p)
    sns.heatmap(cm_t, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_title(title)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
cm_path = os.path.join(LOG_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
print(f"Saved to: {cm_path}")
plt.show()

## 10. Sample Predictions

Green border = correct prediction. Red border = incorrect prediction.

In [ ]:
n_samples = 12
indices   = np.random.choice(len(X_test), n_samples, replace=False)
preds     = (y_pred_prob[indices] > DECISION_THRESHOLD).astype(int)

cols = 4
rows = (n_samples + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
fig.suptitle(f'Sample Predictions — Threshold={DECISION_THRESHOLD} (Green=Correct, Red=Wrong)', fontsize=11)
axes = axes.flatten()

for idx, (i, pred) in enumerate(zip(indices, preds)):
    img   = X_test[i].squeeze()
    true  = class_names[int(y_test[i])]
    p     = class_names[int(pred)]
    color = 'green' if int(y_test[i]) == int(pred) else 'red'
    axes[idx].imshow(img, cmap='gray')
    axes[idx].set_title(f"True: {true}\nPred: {p}", fontsize=9, color=color)
    axes[idx].axis('off')
    for spine in axes[idx].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

for idx in range(n_samples, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
pred_path = os.path.join(LOG_DIR, 'sample_predictions.png')
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
print(f"Saved to: {pred_path}")
plt.show()

## Final Summary

In [ ]:
print("-" * 55)
print("FINAL SUMMARY")
print("-" * 55)
print(f"  Model              : MobileNetV2 + custom head")
print(f"  Data format        : float32 .npy (z-score preserved)")
print(f"  IMG_SIZE           : {IMG_SIZE}")
print(f"  BATCH_SIZE         : {BATCH_SIZE}")
print(f"  Phase 1 LR         : {LEARNING_RATE}")
print(f"  Phase 2 LR         : {FINE_TUNE_LR}")
print(f"  Fine-tune layers   : last {len(base_model.layers) - FINE_TUNE_AT_LAYER}")
print(f"  L2_LAMBDA          : {L2_LAMBDA}")
print(f"  DROPOUT_DENSE      : {DROPOUT_DENSE}")
print(f"  DENSE_UNITS        : {DENSE_UNITS}")
print(f"  DECISION_THRESHOLD : {DECISION_THRESHOLD}")
print(f"  NORMAL weight      : {class_weight_dict[0]:.4f}")
print(f"  PNEUMONIA weight   : {class_weight_dict[1]:.4f}")
print(f"  Model saved        : {MODEL_SAVE_PATH}")
print(f"  Logs               : {LOG_DIR}")
print("\n  Target metrics:")
print("    PNEUMONIA recall  > 0.95  (missing pneumonia is dangerous)")
print("    NORMAL recall     > 0.70  (false alarms are costly)")
print("    AUC               > 0.95  (overall discrimination)")
print("-" * 55)